# Citações verificáveis em RAG

Valide IDs de fonte e citações literais contra o contexto entregue e devolva uma abstenção quando uma citação falhar.

[Artigo completo](../article/ARTICLE.md)

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

cwd = Path.cwd()
lab = next(
    path
    for path in (cwd, cwd.parent, cwd / "0014_citacoes_verificaveis")
    if path.name == "0014_citacoes_verificaveis" and (path / "src" / "main.py").is_file()
)

def run(*options):
    completed = subprocess.run(
        [sys.executable, "-X", "utf8", str(lab / "src" / "main.py"), *options],
        capture_output=True, check=True, text=True, encoding="utf-8",
    )
    return json.loads(completed.stdout)

Execute a próxima célula para reproduzir a comparação e verificar os resultados com asserções.

Os quatro casos têm, em ordem, citação verificada, trecho alterado, fonte ausente e afirmação sem suporte semântico. O último passa na checagem literal. O artigo mostra a diferença entre as duas verificações.

In [ ]:
valid = run("--case", "valid")
altered = run("--case", "altered-quote")
absent = run("--case", "unknown-source")
unsupported = run("--case", "unsupported-claim")
assert valid["status"] == "citations_verified"
assert altered["status"] == "abstained" and altered["errors"][0]["reason"] == "quote_not_found"
assert absent["status"] == "abstained" and absent["errors"][0]["reason"] == "unknown_source"
assert unsupported["status"] == "citations_verified"
for case, outcome in zip(
    ("valid", "altered-quote", "unknown-source", "unsupported-claim"),
    (valid, altered, absent, unsupported),
):
    print(case, outcome["status"], [item["reason"] for item in outcome["errors"]])

## Verifique os trechos selecionados no módulo anterior

Gere o relatório 0013 com dois orçamentos e rode o mesmo caso válido. Os arquivos ficam em uma pasta temporária.

In [ ]:
from tempfile import TemporaryDirectory

selector = lab.parent / "0013_orcamento_contexto" / "src" / "main.py"
observed = {}
with TemporaryDirectory() as directory:
    report_path = Path(directory) / "context.json"
    for budget in (245, 16):
        subprocess.run(
            [sys.executable, "-X", "utf8", str(selector),
             "--bytes", str(budget), "--output", str(report_path)],
            check=True, capture_output=True, encoding="utf-8",
        )
        observed[budget] = run("--case", "valid", "--context-report", str(report_path))
        print(budget, "bytes:", observed[budget]["status"])

assert observed[245]["status"] == "citations_verified"
assert observed[16]["errors"][0]["reason"] == "unknown_source"
print("Com contexto vazio:", observed[16]["errors"][0]["reason"])